In [1]:
import torch
from torchinfo import summary
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from codecarbon import EmissionsTracker
import datetime
import sys
from typing import Union
from scipy.optimize import minimize_scalar

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from core.models.encdec_model import EncDecModel
from core.models.stacked_rnn import Stacked_RNN
from domains.previsao_potencia.dataset import SolarEfficientDataset
from domains.previsao_potencia.preprocessing import SolarPreprocessor
from core.utils.experiment_manager import ExperimentManager
from core.utils.early_stopping import EarlyStopping
from auxiliary_models.brl_diffuse import BRL
from auxiliary_models.kasten_correction import Kasten_Correction
from core.loss_function.cpiloss import CPILoss
from core.loss_function.mseloss import MaskedMSELoss

In [2]:
CONFIG = {
    # --- 1. PATH DA BASE DE DADOS ---
    'csv_path': 'data/pv0.csv',
    
    # --- 2. DIVISÃO DA BASE DE TREINO, TESTE E VALIDAÇÃO ---
    'split_ratios': {'train': 0.8, 'val': 0.2}, 
    'test_year':2022,

    # --- 3. PRÉ-PROCESSAMENTO (Física & Mapeamento) ---
    'preprocessing': {
        'latitude': -23.56,
        'longitude': -46.73,
        'altitude': 0,
        'timezone': 'Etc/GMT+3',
        'nominal_power': 156.0,
        'start_year': 2018,
        'features_to_scale':['temp_amb','wind_speed'],
        #'pv_power_col_csv': 'Pot_BT', # <--- AVALIAR PARA RETIRAR
        
        # DOCUMENTAÇÃO VIVA: Mapeamento "De -> Para"
        # O Preprocessor usará isso para renomear as colunas internamente.
        # Chave (Esquerda): Nome como está no CSV bruto.
        # Valor (Direita): Nome padronizado usado no código.
        'column_mapping': {
            'Pot_BT': 'target',
            'Irradiação Global horária(horizontal) kWh/m2': 'ghi',
            'Irradiação Difusa horária kWh/m2': 'dhi',
            'Irradiação Global horária(Inclinada 27°) kWh/m2': 'irrad_poa',
            'Temperatura ambiente °C': 'temp_amb',
            'Umidade Relativa %': 'humidity',
            'Velocidade média do vento m/s': 'wind_speed'
        }
    },

    # --- 4. ESTRATÉGIA DE MODELAGEM ---
    # mode: 'sky' (prevê k) ou 'power' (prevê kW normalizado)
    'prediction_mode': 'power',
    
    # Qual variável o modelo vai prever? ('kt', 'fracao_difusa' ou 'target')
    'target_col': ['target'], 

    # mascara de dados noturnos
    'use_mask':False,   # True = Ignora a noite (ideal para K)
                        # False = Aprende a noite (ideal para Potência)
    
    # Features de entrada
    'feature_cols': [
        'cos_zenith', 'elevation', 'delta_kt', 
        'delta_fracao_difusa', 'QS', 'wind_speed',
        'temp_amb', 'humidity'
        #'kt', 'fracao_difusa'
        
    ],

    'aux_col':[
        'ghi_cs', 'cos_zenith', 
        'elevation', 'ghi_extra'
    ],

    # --- 5. ARQUITETURA E TREINO ---
    'model_type': 'LSTM',
    'cell_type': 'lstm',
    'input_seq_len': 72,
    'output_seq_len': 1,
    'hidden_sizes': [300, 150],
    'learning_rate': 0.001,
    'batch_size': 32,
    'epochs': 10000,
    'dropout': 0.2,
    'bidirectional': False,
    'use_attention': False,
    'use_feature_attention': False,
    'patience': 100,
    'loss_function':'mse',      # "cpi_loss", "mse", physics_loss
    'physics_base_loss': 'cpi',
    'lambda_hard':10,
    'lambda_soft':10,
}

OUTPUT_ROOT = 'trained_models'
ARTIFACTS_DIR = 'artifacts'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# 1. Setup
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
exp_name = f"{timestamp}_{CONFIG['model_type']}"
exp_dir = os.path.join(OUTPUT_ROOT, exp_name)
os.makedirs(exp_dir, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# 2. Leitura
#csv_path = CONFIG['data/pv0.csv']
#print(f"⏳ Lendo: {csv_path}")
df = pd.read_csv('/workspaces/Remodelacao_mestrado/data/pv0.csv')


if 'Date_Time' in df.columns:
    df['Date_Time'] = pd.to_datetime(df['Date_Time'])
    #df['Date_Time'] -=  pd.Timedelta(minutes=30)
    df = df.drop_duplicates(subset=['Date_Time'], keep='first').set_index('Date_Time').sort_index()
df = df[~df.index.duplicated(keep='first')]

# 3. Pré-processamento
pp_conf = CONFIG['preprocessing']

# Instancia passando o mapa explícito. 
# Isso garante que a padronização aconteça conforme o CONFIG acima.
preprocessor = SolarPreprocessor(
    latitude=pp_conf['latitude'], 
    longitude=pp_conf['longitude'], 
    altitude=pp_conf['altitude'],
    timezone=pp_conf['timezone'], 
    nominal_power=pp_conf['nominal_power'], 
    start_year=pp_conf['start_year'],
    cs_model = 'esra',
    features_to_scale=pp_conf['features_to_scale'],
    target_col=CONFIG['prediction_mode'], # <--- unica variavel que não vem do preprocessing
    column_mapping=pp_conf['column_mapping'],
    kasten_corr=True
)

preprocessor.fit(df)
preprocessor.save_scalers(exp_dir)
preprocessor.save_scalers(ARTIFACTS_DIR)

# O método transform usa o column_mapping para renomear as colunas
df_processed = preprocessor.transform(df)

💾 Scalers salvos em: trained_models/2026-03-05_15-51-30_LSTM
💾 Scalers salvos em: artifacts
Otimizando TL para 284 dias selecionados...
Otimização concluída. Média TL: 6.30


In [4]:
df_processed

,target,ghi,dhi,irrad_poa,temp_amb,humidity,wind_speed,Pressão Baromêtrica mm Hg,Pluviômetro mm,zenith,...,k,QS,VS_cdfn,temp_cell,pot_cs,P1,P2,P3,irr_clearsky_ratio,mask
Date_Time,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00-03:00,0.000000,0.0,0.0,0.0,0.699062,0.845,0.119115,762.00,0.0,133.365508,...,0.000000,0.292893,0.0,21.86,29.928621,0.000000,0.000000,0.000000,0.000000,0.0
2018-01-01 01:00:00-03:00,0.000000,0.0,0.0,0.0,0.694429,0.860,0.076271,761.00,0.0,131.898130,...,0.000000,0.292893,0.0,21.45,29.983070,0.000000,0.000000,0.000000,0.000000,0.0
2018-01-01 02:00:00-03:00,0.000000,0.0,0.0,0.0,0.691378,0.875,0.049906,761.00,0.0,126.359488,...,0.000000,0.292893,0.0,21.18,30.018927,0.000000,0.000000,0.000000,0.000000,0.0
2018-01-01 03:00:00-03:00,0.000000,0.0,0.0,0.0,0.686857,0.892,0.048493,761.00,0.0,117.831752,...,0.000000,0.292893,0.0,20.78,30.072047,0.000000,0.000000,0.000000,0.000000,0.0
2018-01-01 04:00:00-03:00,0.000000,0.0,0.0,0.0,0.685727,0.894,0.057910,760.00,0.0,107.365526,...,0.000000,0.292893,0.0,20.68,30.085328,0.000000,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-31 19:00:00-03:00,0.000371,29.0,24.0,25.0,0.699062,0.922,0.000000,700.04,0.0,91.482672,...,0.062615,0.082150,0.0,23.02,72.894515,0.082532,0.119241,0.137413,0.862069,0.0
2022-12-31 20:00:00-03:00,0.000000,0.0,0.0,0.0,0.695220,0.884,0.000942,700.66,0.0,103.460544,...,0.000000,0.292893,0.0,21.52,73.382207,0.000373,0.083084,0.120039,0.000000,0.0
2022-12-31 21:00:00-03:00,0.000000,0.0,0.0,0.0,0.684145,0.926,0.000471,700.71,0.0,114.404647,...,0.000000,0.292893,0.0,20.54,73.702982,0.000000,0.000375,0.083447,0.000000,0.0
